# Task 3: Model Development

**Objective:** Build and train machine learning models for flight delay prediction using PySpark ML, implementing a complete pipeline with StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, and four classification algorithms as specified in the assignment requirements.

This notebook focuses on model development, including pipeline creation, model training with cross-validation, hyperparameter tuning, and performance evaluation using the prepared dataset from Task 2.

## Methodology

1. Load the processed dataset from Task 2
2. Split data into training and test sets (80/20 split)
3. Build a reusable PySpark ML Pipeline with:
   - StringIndexer for categorical features
   - OneHotEncoder for categorical variables
   - VectorAssembler to combine features
   - StandardScaler for feature normalization
   - Classification algorithm ( Logistic Regression, Decision Tree, Random Forest, GBTClassifier)
4. Implement CrossValidator with ParamGridBuilder for hyperparameter tuning
5. Train and compare four models:
   - Logistic Regression
   - Decision Tree
   - Random Forest
   - GBTClassifier
6. Use 3-fold cross-validation as specified
7. Calculate performance metrics: Accuracy, Precision, Recall, F1-score, AUC-ROC
8. Create a summary table comparing all models
9. Provide interpretation notes for academic report

In [ ]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import time

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("7006SCN-Task3-ModelDevelopment") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Set log level
spark.sparkContext.setLogLevel("WARN")

# Load processed dataset from Task 2
DATA_PATH = "data/On_Time_Reporting_Carrier_On_Time_Performance_2024_1.csv"
print("Loading processed dataset...")

# Recreate the data processing steps from Task 2 to get the processed dataset
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("nullValue", "") \
    .option("treatEmptyValuesAsNulls", "true") \
    .csv(DATA_PATH)

# Apply the same data preparation steps as Task 2
df_clean = df.dropDuplicates()

# Handle missing values
numerical_cols = [field.name for field in df_clean.schema.fields if isinstance(field.dataType, (IntegerType, LongType, FloatType, DoubleType))]
string_cols = [field.name for field in df_clean.schema.fields if isinstance(field.dataType, StringType)]

df_filled = df_clean
for col_name in numerical_cols:
    if col_name in df_clean.columns:
        median_val = df_clean.approxQuantile(col_name, [0.5], 0.25)[0]
        if median_val is not None:
            df_filled = df_filled.fillna({col_name: median_val})
        else:
            df_filled = df_filled.fillna({col_name: 0})

for col_name in string_cols:
    if col_name in df_clean.columns:
        df_filled = df_filled.fillna({col_name: "Unknown"})

# Remove cancelled and diverted flights
df_filtered = df_filled.filter((col("Cancelled") == 0) & (col("Diverted") == 0))

# Remove leakage variables
leakage_vars = ["ArrDelay", "ArrDelayMinutes", "ArrivalDelayGroups", "ArrTime", "ArrTimeBlk"]
existing_leakage = [var for var in leakage_vars if var in df_filtered.columns]
df_no_leakage = df_filtered.drop(*existing_leakage)

# Remove identifier columns
identifier_cols = ["Tail_Number", "Flight_Number_Reporting_Airline", "FlightDate"]
existing_identifiers = [col for col in identifier_cols if col in df_no_leakage.columns]
df_clean = df_no_leakage.drop(*existing_identifiers)

# Select features as specified
numeric_features = ["Month", "DayOfWeek", "CRSDepTime", "DepDelay", "TaxiOut", "Distance", "AirTime"]
categorical_features = ["Reporting_Airline", "Origin", "Dest"]
label_column = "ArrDel15"
required_columns = numeric_features + categorical_features + [label_column]
df_selected = df_clean.select(required_columns)

# Apply stratified sampling (15% of each class) as specified
sampled_df = df_selected.sampleBy(
    label_column,
    fractions={0: 0.15, 1: 0.15},
    seed=42
)

print(f"Processed dataset loaded: {sampled_df.count():,} rows, {len(sampled_df.columns)} columns")

# Split data into training and test sets (80/20)
train_data, test_data = sampled_df.randomSplit([0.8, 0.2], seed=42)
print(f"Training set: {train_data.count():,} rows")
print(f"Test set: {test_data.count():,} rows")

# Cache training and test data for performance
train_data.cache()
test_data.cache()

## Pipeline Functions

Create reusable functions for building machine learning pipelines as required.

In [ ]:
# Define feature sets
numeric_features = ["Month", "DayOfWeek", "CRSDepTime", "DepDelay", "TaxiOut", "Distance", "AirTime"]
categorical_features = ["Reporting_Airline", "Origin", "Dest"]
label_column = "ArrDel15"

def create_pipeline(classifier):
    """
    Create a reusable PySpark ML Pipeline with specified classifier.
    
    Args:
        classifier: PySpark ML classifier object
    
    Returns:
        Pipeline: Configured machine learning pipeline
    """
    # StringIndexer for categorical features
    string_indexers = [
        StringIndexer(inputCol=cat, outputCol=f"{cat}_indexed")
        for cat in categorical_features
    ]

    # OneHotEncoder for categorical variables
    one_hot_encoders = [
        OneHotEncoder(inputCol=f"{cat}_indexed", outputCol=f"{cat}_encoded")
        for cat in categorical_features
    ]

    # VectorAssembler to combine all features
    indexed_categorical_cols = [f"{cat}_indexed" for cat in categorical_features]
    encoded_categorical_cols = [f"{cat}_encoded" for cat in categorical_features]
    all_features = numeric_features + encoded_categorical_cols
    assembler = VectorAssembler(inputCols=all_features, outputCol="features")

    # StandardScaler for feature normalization
    scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)

    # Create pipeline stages
    stages = string_indexers + one_hot_encoders + [assembler, scaler, classifier]
    
    return Pipeline(stages=stages)

def get_param_grids():
    """
    Define parameter grids for hyperparameter tuning as specified in assignment.
    
    Returns:
        dict: Parameter grids for each classifier
    """
    param_grids = {
        "Logistic Regression": ParamGridBuilder() \
            .addGrid(LogisticRegression().regParam, [0.01, 0.1]) \
            .addGrid(LogisticRegression().elasticNetParam, [0.0, 0.5]) \
            .build(),
        
        "Decision Tree": ParamGridBuilder() \
            .addGrid(DecisionTreeClassifier().maxDepth, [5, 10]) \
            .addGrid(DecisionTreeClassifier().maxBins, [32, 64]) \
            .build(),
        
        "Random Forest": ParamGridBuilder() \
            .addGrid(RandomForestClassifier().numTrees, [50, 100]) \
            .addGrid(RandomForestClassifier().maxDepth, [8, 12]) \
            .build(),
    
        "GBT Classifier": ParamGridBuilder() \
            .addGrid(GBTClassifier().maxIter, [20, 50]) \
            .addGrid(GBTClassifier().maxDepth, [5, 8]) \
            .build()
    }
    
    return param_grids

def evaluate_model(model, test_data, label_col):
    """
    Evaluate a trained model and return performance metrics.
    
    Args:
        model: Trained PySpark ML model
        test_data: Test dataset
        label_col: Name of the label column
    
    Returns:
        dict: Dictionary containing accuracy, precision, recall, f1, and auc
    """
    # Make predictions
    predictions = model.transform(test_data)

    # Initialize evaluators
    binary_evaluator = BinaryClassificationEvaluator(
        labelCol=label_col, 
        rawPredictionCol="rawPrediction", 
        metricName="areaUnderROC"
    )
    
    multi_evaluator = MulticlassClassificationEvaluator(
        labelCol=label_col, 
        predictionCol="prediction"
    )
    
    # Calculate metrics
    auc = binary_evaluator.evaluate(predictions)
    accuracy = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "accuracy"})
    precision = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "weightedPrecision"})
    recall = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "weightedRecall"})
    f1 = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "weightedFMeasure"})
    
    return {
        "accuracy": accuracy,
    
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

## Model Training and Evaluation

Train and compare the four specified models using cross-validation and hyperparameter tuning.

In [ ]:
# Get parameter grids
param_grids = get_param_grids()

# Define classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(featuresCol="scaledFeatures", labelCol=label_column, maxIter=100),
    "Decision Tree": DecisionTreeClassifier(featuresCol="scaledFeatures", labelCol=label_column),
    "Random Forest": RandomForestClassifier(featuresCol="scaledFeatures", labelCol=label_column),
    "GBT Classifier": GBTClassifier(featuresCol="scaledFeatures", labelCol=label_column)
}

# Dictionary to store results
model_results = {}
trained_models = {}

# Train and evaluate each model
for model_name, classifier in classifiers.items():
    print(f"\n{'='*60}")
    print(f"Training {model_name}...")
    print(f"{'='*60}")

    start_time = time.time()

    # Create pipeline
    pipeline = create_pipeline(classifier)

    # Set up cross-validation
    crossval = CrossValidator(
        estimator=pipeline,
    
        estimatorParamMaps=param_grids[model_name],
        evaluator=BinaryClassificationEvaluator(labelCol=label_column, rawPredictionCol="rawPrediction", metricName="areaUnderROC"),
        numFolds=3,
        seed=42
    )

    # Train model
    cv_model = crossval.fit(train_data)
    
    training_time = time.time() - start_time
    
    # Get best model
    best_model = cv_model.bestModel
    
    # Evaluate model
    metrics = evaluate_model(best_model, test_data, label_column)
    metrics["training_time"] = training_time
    
    # Store results
    model_results[model_name] = metrics
    trained_models[model_name] = best_model
    
    # Print results
    print(f"Results for {model_name}:")
    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    print(f"  F1-Score: {metrics['f1']:.4f}")
    print(f"  AUC-ROC: {metrics['auc']:.4f}")
    print(f"  Training Time: {training_time:.2f} seconds")
    
    # For tree-based models, show feature importance
    if hasattr(best_model.stages[-1], "featureImportances"):
        print(f"  Feature Importances: {best_model.stages[-1].featureImportances}")

    # Clean up
    del cv_model

## Model Performance Summary

Create a summary table comparing all models as required.

In [ ]:
# Create summary table
print(f"\n{'='*80}")
print("MODEL PERFORMANCE SUMMARY")
print(f"{'='*80}")
print(f"{"Model":<20} {"Accuracy":<10} {"Precision":<10} {"Recall":<10} {"F1":<10} {"AUC":<10} {"Time (s)":<10}")
print(f"{'-'*80}")

for model_name, metrics in model_results.items():
    print(f"{model_name:<20} {metrics['accuracy']:<10.4f} {metrics['precision']:<10.4f} {metrics['recall']:<10.4f} {metrics['f1']:<10.4f} {metrics['auc']:<10.4f} {metrics['training_time']:<10.2f}")

print(f"{'-'*80}")

# Find best model based on AUC
best_model_name = max(model_results, key=lambda x: model_results[x]['auc'])
best_auc = model_results[best_model_name]['auc']
print(f"\n🏆 Best performing model (by AUC): {best_model_name}")
print(f"   AUC: {best_auc:.4f}")

# Save the best model
MODEL_PATH = "models/best_flight_delay_model"
os.makedirs("models", exist_ok=True)
trained_models[best_model_name].save(MODEL_PATH)
print(f"\nBest model saved to: {MODEL_PATH}")

# Save all models for completeness
for model_name, model in trained_models.items():
    model_path = f"models/{model_name.replace(' ', '_').replace('-', '_')}_model"
    model.save(model_path)
    print(f"Model '{model_name}' saved to: {model_path}")

# Unpersist cached data
train_data.unpersist()
test_data.unpersist()
sampled_df.unpersist()
print("\nCached data unpersisted")